# Spark et Databricks (Partie 1) : ton premier notebook PySpark

📖 **Le mini-cours complet** : [Spark et Databricks : du big data à ton premier notebook PySpark (Partie 1)](https://www.monshotdata.com/p/spark-databricks-calcul-distribue-partie-1)

🧭 **Le cours** : [Bonnes pratiques d'ingénierie › 1.4 Passage à l'échelle et calcul distribué](https://www.monshotdata.com/mini-cours/bonnes-pratiques-ingenierie#h2sc14)

---

Pandas et Scikit-learn supposent que toutes tes données tiennent dans la RAM d'une seule machine. Au-delà, il faut un moteur distribué : Spark. Ici tu fais tes premiers pas sur **Databricks Free Edition** : tu vérifies ta session Spark, tu ranges un fichier dans un Volume et tu le lis avec PySpark.

**À la fin de ce notebook, tu sauras :**

- vérifier la session Spark que Databricks crée pour toi, et sa version ;
- créer un **Volume** Unity Catalog et y écrire un fichier CSV ;
- lire ce fichier avec `spark.read` et afficher un DataFrame Spark.

### Avant de lancer

Ce notebook tourne sur **Databricks**, pas sur Colab. Un compte **Databricks Free Edition** (gratuit) suffit.

- **Ouvrir le notebook** : dans Databricks, Workspace → Import, puis dépose ce fichier `.ipynb` (ou colle l'URL du fichier sur GitHub).
- **Le lancer** : attache-le au calcul **serverless**, puis exécute les cellules de haut en bas (ou « Run all »).
- **Les données** : le notebook crée un Volume `workspace.default.cours_spark` et y écrit `employes.csv`, un dataset fictif de 1 000 employés (graine fixe : tu obtiens les mêmes chiffres). Les Parties 2 et 3 réutilisent ce fichier.

🧪 *Testé le 21 novembre 2026 : exécution complète de haut en bas sur Databricks Free Edition (calcul serverless, Spark 4.2, Python 3.12). Les sorties affichées viennent de cette exécution ; les identifiants techniques (adresse de la session, noms de fichiers `part-…`, `uid` des modèles) changent à chaque lancement. La sortie de `display()` est réduite ici aux 6 premières lignes pour garder le fichier léger : dans Databricks, tu verras le tableau interactif complet.*

## 1. Ton premier notebook PySpark sur Databricks Free Edition

Vérifie la session Spark, crée un Volume avec un CSV de 1 000 employés fictifs, puis lis-le avec Spark.

In [1]:
# La session Spark est déjà créée par Databricks : elle s'appelle spark
print(spark)
print("Version de Spark :", spark.version)

# Résultat :
# <pyspark.sql.connect.session.SparkSession object at 0xffc48269d520>
# Version de Spark : 4.2.0

Version de Spark : 4.2.0


In [2]:
import numpy as np
import pandas as pd

# Crée un Volume : un espace de stockage de fichiers dans Unity Catalog
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.cours_spark")
chemin_volume = "/Volumes/workspace/default/cours_spark"

# Dataset fictif de 1 000 employés (graine fixe : résultats reproductibles)
rng = np.random.default_rng(42)
n = 1000
prenoms = ["Léa", "Hugo", "Emma", "Louis", "Chloé",
           "Lucas", "Inès", "Nathan", "Jade", "Adam"]
noms = ["Martin", "Bernard", "Dubois", "Thomas", "Robert",
        "Richard", "Petit", "Durand", "Leroy", "Moreau"]
villes = ["Paris", "Lyon", "Marseille", "Toulouse", "Bordeaux", "Lille"]
postes = ["Data Analyst", "Data Scientist", "Data Engineer",
          "ML Engineer", "Manager"]

employes = pd.DataFrame({
    "id_employe": np.arange(1001, 1001 + n),
    "prenom": rng.choice(prenoms, n),
    "nom": rng.choice(noms, n),
    "age": rng.integers(22, 61, n),
    "ville": rng.choice(villes, n),
    "poste": rng.choice(postes, n),
    "salaire": rng.integers(32, 95, n) * 1000,
})

# Écrit le fichier CSV dans le Volume
employes.to_csv(f"{chemin_volume}/employes.csv", index=False)
print(employes.shape)

# Résultat :
# (1000, 7)

(1000, 7)


In [3]:
# Lis le CSV avec Spark : la première
# ligne contient les noms de colonnes
df_employes = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{chemin_volume}/employes.csv")
)
display(df_employes)

id_employe,prenom,nom,age,ville,poste,salaire
1001,Léa,Leroy,56,Bordeaux,Data Scientist,44000
1002,Nathan,Durand,24,Bordeaux,Manager,34000
1003,Inès,Richard,37,Lille,Data Analyst,71000
1004,Chloé,Leroy,39,Lille,Data Engineer,75000
1005,Chloé,Bernard,53,Bordeaux,Manager,80000
1006,Jade,Moreau,27,Marseille,Manager,91000


In [4]:
# Schéma déduit par Spark et nombre de lignes
df_employes.printSchema()
print(df_employes.count(), "lignes")

# Résultat :
# root
#  |-- id_employe: integer (nullable = true)
#  |-- prenom: string (nullable = true)
#  |-- nom: string (nullable = true)
#  |-- age: integer (nullable = true)
#  |-- ville: string (nullable = true)
#  |-- poste: string (nullable = true)
#  |-- salaire: integer (nullable = true)
#
# 1000 lignes

root
 |-- id_employe: integer (nullable = true)
 |-- prenom: string (nullable = true)
 |-- nom: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- ville: string (nullable = true)
 |-- poste: string (nullable = true)
 |-- salaire: integer (nullable = true)

1000 lignes


## 2. Ce qu'il faut retenir

- Pandas et Scikit-learn supposent que tout tient dans la RAM d'une seule machine : au-delà, il faut un système distribué comme Spark.
- Spark remplace MapReduce, pas Hadoop : sa force vient du traitement en mémoire, à condition d'éviter le disk spill.
- Les RDD sont immuables, distribués et tolérants aux pannes grâce à leur lineage ; en pratique, tu les manipules via l'API DataFrame.
- Les transformations sont paresseuses : rien ne s'exécute avant une action (afficher, écrire, ramener des données côté Python).
- Databricks Free Edition te donne un Spark serverless prêt à l'emploi : choisis l'environnement le plus récent et range tes fichiers dans un Volume.

---

## Pour aller plus loin

- [Spark et Databricks : créer, lire et écrire des DataFrames PySpark (Partie 2)](https://www.monshotdata.com/p/spark-databricks-dataframes-pyspark-partie-2) — la partie suivante : créer, lire, écrire et afficher des DataFrames